# Consistency: a label-free judge from corrupted reasoning (offline test): steps 1 and 2

Good reasoning tends to give good solutions, broken reasoning bad ones. The SFT policy samples normal completions; the
reasoning of each is then corrupted, and the policy writes only the solution + answer from it, with just enough room
for a solution:
- **empty**: no reasoning at all
- **swap**: the reasoning of another puzzle (same size)
- **cut**: the first `CUT_FRACTION` of the trace's own reasoning (whole lines), so the cut scales with each trace
- **hot** (`HOT_TEMPERATURES`): a new reasoning for the same puzzle, sampled hot (meant to keep the format and make
  the search sloppier); kinds "hot1.3", "hot1.5"

DPO judge (the current judge's recipe, only the pairs change): chosen = the normal solution, rejected = a solution from
corrupted reasoning, same puzzle (training: the TRAIN_KINDS pairs). No checker labels in training; the checker only
measures.

This notebook samples (vLLM); Consistency_Corrupt_Judge trains and tests the judge on what it saves, in a fresh
runtime (vLLM keeps its GPU memory).

Run order: Run all. [generate] skips what is
saved (each hot temperature has its own file), [label gap] shows every kind, [release] frees the runtime
(UNASSIGN_WHEN_DONE).

In [1]:
# [config] Everything you choose lives here.
MODEL = "meta-llama/Llama-3.2-3B-Instruct"
SFT_RUN = "Llama-3.2-3B-Instruct-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-judge500-mix3x500"
# the GRPO start (trace v4 without unsolvable traces, small SFT, 2026-10-26; v4 with them: -imp38x38); the 2026-10-23/24 offline test used the mixed SFT, OLD_SFT_RUN
OLD_SFT_RUN = "Llama-3.2-3B-Instruct-countdown3000-ep1-lr0.0001-seed0-tr2-judge3000-mix3x3000"
SOLVER_DPO_RUN = "judge_dpo_swap30extra30slip20miss20_n2250_n3x2250_b0.1_lr2e-05_ep1"     # the solver-pair judge (comparison;
                                         # trained by Consistency_Countdown's [judge DPO] under SFT_DIR)
DRIVE = "/content/drive/MyDrive/Consistency"
CODE_DIR = f"{DRIVE}/code/countdown"
SFT_DIR = f"{DRIVE}/countdown/sft/{SFT_RUN}"
ADAPTER_PATH = f"{SFT_DIR}/adapter"                          # the policy that samples, and the DPO start and reference
SOLVER_JUDGE_PATH = f"{SFT_DIR}/{SOLVER_DPO_RUN}/adapter"   # "solver": the solver-pair judge of THIS SFT (2026-10-26)
SOLVER_OLD_JUDGE_PATH = f"{DRIVE}/countdown/sft/{OLD_SFT_RUN}/{SOLVER_DPO_RUN}/adapter"   # "solver_old": the mixed SFT's,
# which scored the GRPO dumps (their logged judge_mean: [dump probe]'s check); differs from the corrupt judge in pair
# type AND SFT start
# Step 1: normal completions, one per puzzle, as GRPO samples them. Puzzles: a new seeded draw (SEED + 6) without every
# puzzle of the test, SFT, GRPO RL and GRPO 3-number draws (Consistency_Countdown's real-pair exclusion) and without the
# dump puzzles; the last VAL_SHARE of each size give the validation pairs
N_PUZZLES_4, N_PUZZLES_3 = 1000, 1000
VAL_SHARE = 0.1
TEMPERATURE = 0.9
MAX_TOKENS = 1024
SEED = 0
# Step 2: corruptions of each normal completion's reasoning; the policy then writes <solution> and <answer> only
KINDS = ("empty", "swap", "cut")
HOT_TEMPERATURES = (1.3, 1.5)            # hot: a new reasoning at each temperature (kind "hot1.3", ...), own file
CUT_FRACTION = 0.5                       # cut: keep this share of the reasoning's tokens, back to the last whole line
SOL_MAX_TOKENS = None                    # room for solution + answer; None = the 99th percentile over the normal
                                         # completions (from "<solution>" through "</answer>"), the same cap for both
PLAUSIBILITY_MAX_SOLUTION_TOKENS = 128   # as the GRPO reward: longer solutions, or no answer, are not scored
VLLM_GPU_MEMORY_UTILIZATION = 0.5
# Step 3: DPO as the current judge ([judge DPO] of Consistency_Countdown): continue the SFT adapter, reference = the SFT
# adapter, batch 16
DPO_BETA, DPO_LR, DPO_EPOCHS = 0.1, 2e-5, 1
TRAIN_KINDS = ("empty", "cut")          # the corruptions whose pairs train the judge, e.g. ("hot1.3",) (validation:
                                         # every kind, hot ones included)
RETRAIN_DPO = False
# Step 4: the GRPO dumps (as Consistency_Judge_Score_Probe): the mean-34 run from the old mixed SFT (the new SFT has no
# GRPO runs yet), so the judge is tested on another policy's fakes; also excluded from the step-1 puzzles
GRPO_DIR = f"{DRIVE}/countdown/grpo/{OLD_SFT_RUN}-judge_dpo-start_sft"
RUN = f"{GRPO_DIR}/runs/grpo-mean-missmed-mix3_0.5-expr-kl0.02-swap30extra30slip20miss20_n2250_n3x2250-mean-34"
DUMP_DIRS = [(RUN, 0, 99), (RUN + "-cont100b", 100, 10**6)]   # (run, first step, last step)
CHUNK = 8                                # sequences per scoring forward pass
UNASSIGN_WHEN_DONE = True                # release the Colab runtime after the last cell (stops the compute bill)

OUT_DIR = (f"{SFT_DIR}/judge_corrupt_{'-'.join(KINDS)}" + (f"_cut{CUT_FRACTION:g}" if "cut" in KINDS else "")
           + f"_n4x{N_PUZZLES_4}_n3x{N_PUZZLES_3}_t{TEMPERATURE:g}_seed{SEED}")
NORMAL_PATH, CORRUPT_PATH = f"{OUT_DIR}/normal.csv", f"{OUT_DIR}/corrupt.csv"
HOT_PATHS = {t: f"{OUT_DIR}/corrupt_hot{t:g}.csv" for t in HOT_TEMPERATURES}
ALL_KINDS = (*KINDS, *(f"hot{t:g}" for t in HOT_TEMPERATURES))
_dpo = f"b{DPO_BETA:g}_lr{DPO_LR:g}_ep{DPO_EPOCHS}"
JUDGE_DIR = f"{OUT_DIR}/dpo_" + ("" if set(TRAIN_KINDS) == set(KINDS) else f"{'-'.join(TRAIN_KINDS)}_") + _dpo
ALL_KINDS_JUDGE_PATH = f"{OUT_DIR}/dpo_{_dpo}/adapter"   # trained on every kind in KINDS (a comparison, if it exists)
JUDGE_PATH = f"{JUDGE_DIR}/adapter"
assert set(KINDS) <= {"empty", "swap", "cut"} and set(TRAIN_KINDS) <= set(ALL_KINDS), (KINDS, TRAIN_KINDS)
print(f"OUT_DIR={OUT_DIR}\nJUDGE_DIR={JUDGE_DIR}")

OUT_DIR=/content/drive/MyDrive/Consistency/countdown/sft/Llama-3.2-3B-Instruct-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-judge500-mix3x500/judge_corrupt_empty-swap-cut_cut0.5_n4x1000_n3x1000_t0.9_seed0
JUDGE_DIR=/content/drive/MyDrive/Consistency/countdown/sft/Llama-3.2-3B-Instruct-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-judge500-mix3x500/judge_corrupt_empty-swap-cut_cut0.5_n4x1000_n3x1000_t0.9_seed0/dpo_empty-cut_b0.1_lr2e-05_ep1


In [2]:
# [drive + HF] Mount Drive and log in to HF (Colab Secret HF_TOKEN; Llama is gated). Re-running is harmless.
import os
from google.colab import drive, userdata
if not os.path.ismount("/content/drive"):
      drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # read by transformers and vLLM; no huggingface_hub import before [install]
for f in (f"{ADAPTER_PATH}/adapter_model.safetensors", f"{SOLVER_JUDGE_PATH}/adapter_model.safetensors",
          f"{SOLVER_OLD_JUDGE_PATH}/adapter_model.safetensors",
          f"{CODE_DIR}/countdown.py", f"{CODE_DIR}/prompts.py", f"{CODE_DIR}/rewards.py", f"{CODE_DIR}/judge_data.py"):
    assert os.path.exists(f), f"missing {f}"

Mounted at /content/drive


In [3]:
%pip install -U "trl[vllm]" peft
%pip uninstall -y torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 61.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 MB 101.4 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 124.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 153.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 153.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 132.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 80.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 MB 125.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.3/88.3 MB 127.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 47.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 115.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 156.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.6/729.6 kB 46.8 MB/s  0:00:00
   ━━

In [4]:
# [install] Same packages as Consistency_Countdown (vLLM samples), once per runtime: skipped when vllm and trl are
# installed. No restart: nothing they replace (torch, transformers) is imported before this cell.
import importlib.util, subprocess, sys
if importlib.util.find_spec("vllm") and importlib.util.find_spec("trl"):
    print("[install] skipped: vllm and trl are installed")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "trl[vllm]", "peft"], check=True)
    print("[install] done")
if importlib.util.find_spec("torchao"):   # removed either way: Colab ships 0.10, peft refuses to import below 0.16
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
    print("[install] removed torchao")

[install] skipped: vllm and trl are installed


In [5]:
# [vLLM setup] Run once per session BEFORE anything touches the GPU.
import os, sys
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"  # keep the vLLM engine in this process
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
for stream, fd in ((sys.stdout, 1), (sys.stderr, 2)):   # Colab's streams have no file descriptor; vLLM asks for one
    try:
        stream.fileno()
    except Exception:
        stream.fileno = lambda fd=fd: fd

!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import importlib.metadata as md
for p in ["torch", "transformers", "datasets", "vllm"]:
    try:
        print(f"{p:14s}", md.version(p))
    except md.PackageNotFoundError:
        print(f"{p:14s}", "not installed")

name, memory.total [MiB], memory.used [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB, 0 MiB
torch          2.13.0
transformers   5.17.0
datasets       4.8.5
vllm           0.31.0


In [6]:
# [setup] Tokenizer, the Countdown code (Drive), the GRPO dumps (step 4, and excluded from the puzzles), the puzzles, and
# the helpers: the judge's view of a completion (as the GRPO reward scores it) and what kind of answer it is.
import os, sys, re, glob, json, random, importlib, math, gc
from collections import Counter
from fractions import Fraction
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = "<|finetune_right_pad_id|>"   # not <|eot_id|> (the EOS)
tok.chat_template = re.sub(r'strftime_now\([^)]*\)', '"26 Jul 2024"', tok.chat_template)   # pinned date

def chat_ids(messages):
    """Templated prompt + the assistant header as token ids, no extra BOS."""
    return tok(tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True),
               add_special_tokens=False)["input_ids"]

sys.path.insert(0, CODE_DIR)
import prompts, countdown, traces, rewards, judge_data
for _m in (prompts, countdown, traces, rewards, judge_data):   # prompts first: the others import from it
    importlib.reload(_m)
from countdown import check_answer
from prompts import countdown_prompt, bare_question, judge_messages, NEUTRAL_SYSTEM_PROMPT
from rewards import tag, diagnose
assert judge_messages([1, 2, 3, 4], 10) == [{"role": "system", "content": NEUTRAL_SYSTEM_PROMPT},
                                            {"role": "user", "content": bare_question(countdown_prompt([1, 2, 3, 4], 10))}], \
    "judge prompt differs from the GRPO reward's"

_NUM = r"\(?\s*(-?\d+(?:\.\d+)?(?:\s*/\s*\d+)?)\s*\)?"      # Consistency_Countdown's check + decimals: (-5), 3/4, 6.5
_STEP = re.compile(rf"^\s*{_NUM}\s*([+\-*/×÷x])\s*{_NUM}\s*=\s*{_NUM}\s*\.?\s*$")

def replay(sol, numbers):
    """Replays the solution's steps from the given numbers: "false" (a step's result is wrong), "unavailable" (every
    step true, an operand is neither a given number nor an earlier result, or is used twice), "valid" (and the last
    result), or "unparsed" (a line that is not "a op b = c", after dropping a trailing "Left: ...")."""
    avail = Counter(Fraction(x) for x in numbers)
    bad = False
    lines = [l for l in (sol or "").splitlines() if l.strip()]
    if not lines:
        return "unparsed"
    for line in lines:
        m = _STEP.match(line.split("Left:")[0])
        if not m:
            return "unparsed"
        a, op, b, c = m.groups()
        a, b, c = (Fraction(s.replace(" ", "")) for s in (a, b, c))
        r = {"+": a + b, "-": a - b, "*": a * b, "×": a * b, "x": a * b}.get(op, a / b if b else None)
        if r != c:
            return "false"
        for v in (a, b):
            bad |= avail[v] < 1
            avail[v] -= 1
        avail[c] += 1
    return "unavailable" if bad else ("valid", c)

def klass(sol, ans, numbers, target):
    """correct / fake (a false step in the solution) / wrong numbers (every step true, a number made up or reused) /
    honest miss (a valid route from the given numbers that misses the target) / bad answer (a valid route to the
    target, the answer gets it wrong) / other (a solution line we can't parse) / no answer."""
    if not isinstance(ans, str) or not ans.strip():
        return "no answer"
    if check_answer(ans, list(numbers), int(target)):
        return "correct"
    r = replay(sol, numbers)
    if isinstance(r, tuple):
        return "bad answer" if r[1] == target else "honest miss"
    return {"false": "fake", "unavailable": "wrong numbers", "unparsed": "other"}[r]

def scorable(sol, ans):
    """Scored by the GRPO reward: a solution of at most PLAUSIBILITY_MAX_SOLUTION_TOKENS tokens and an answer."""
    return (isinstance(sol, str) and bool(sol.strip()) and isinstance(ans, str) and bool(ans.strip())
            and len(tok(sol, add_special_tokens=False)["input_ids"]) <= PLAUSIBILITY_MAX_SOLUTION_TOKENS)

judge_text = lambda sol, ans: f"{sol.strip()}\nAnswer: {ans.strip()}"

# The GRPO dumps (Consistency_Judge_Score_Probe's [data]): step 4's test set
rows = []
for run, lo, hi in DUMP_DIRS:
    for f in sorted(glob.glob(f"{run}/step_*.txt")):
        step = int(re.search(r"step_(\d+)", f).group(1))
        if not lo <= step <= hi:
            continue
        for blk in open(f).read().split("### Sample ")[1:]:
            sc = re.search(r"puzzle=(\d+) correct=(\w+) judge_mean=(\S+)", blk)
            prompt = blk.split("[Prompt]\n", 1)[1].split("\n\n[Response]\n", 1)[0]
            resp = blk.split("[Response]\n", 1)[1].split("\n\n" + "=" * 20, 1)[0]
            pm = re.match(r"Using the numbers (.*?), create an expression that equals (\d+)\.", prompt)
            nums, tgt = [int(x) for x in pm.group(1).split(", ")], int(pm.group(2))
            assert countdown_prompt(nums, tgt)[-1]["content"] == prompt, f"prompt differs: {f}"
            sol, ans = tag(resp, "solution"), tag(resp, "answer")
            rows.append({"step": step, "pid": int(sc.group(1)), "n": len(nums), "numbers": nums, "target": tgt,
                         "correct": sc.group(2) == "True", "judge_logged": float(sc.group(3)),
                         "solution": sol, "answer": ans, "kind": klass(sol, ans, nums, tgt)})
dumps = pd.DataFrame(rows)
dumps["group"] = dumps.step.astype(str) + "/" + dumps.pid.astype(str)
assert len(dumps) and (dumps.groupby("group").size() == 16).all(), "no dumps, or a group without 16 completions"
_dis = ((dumps.kind == "correct") != dumps.correct).mean()
assert _dis < 0.01, f"klass and the logged checker disagree on {_dis:.1%} of the dumps"
print(f"dumps: {len(dumps)} completions, {dumps.group.nunique()} groups, steps {sorted(dumps.step.unique())}")

# Puzzles: Consistency_Countdown's real-pair exclusion (test draw, SFT draw, GRPO RL and 3-number draws at the GRPO
# defaults), a new seed, and none of the dump puzzles
_tz = load_dataset("Jiayi-Pan/Countdown-Tasks-3to4", split="train")
_as = lambda r: {"numbers": [int(x) for x in r["nums"]], "target": int(r["target"])}
_key = lambda p: (tuple(sorted(p["numbers"])), p["target"])
_seeded = lambda seed, size: _tz.select(np.random.default_rng(seed).choice(len(_tz), size=min(len(_tz), size), replace=False))
_no = {_key(_as(r)) for seed, size in ((SEED, 20 * 200), (SEED + 1, 6 * (4000 + 100)), (SEED + 2, 4 * (4000 + 100)),
                                        (SEED + 3, 4 * 3900)) for r in _seeded(seed, size)}
_no |= {_key(r) for r in dumps[["numbers", "target"]].to_dict("records")}
_want = {4: N_PUZZLES_4, 3: N_PUZZLES_3}
_got = {4: [], 3: []}
for r in _seeded(SEED + 6, 10 * (N_PUZZLES_4 + N_PUZZLES_3)):
    p = _as(r)
    n = len(p["numbers"])
    if _key(p) not in _no and len(_got[n]) < _want[n]:
        _no.add(_key(p))
        _got[n].append(p)
assert all(len(_got[n]) == _want[n] for n in _want), {n: len(v) for n, v in _got.items()}
puzzles = []
for n in (4, 3):
    n_val = round(VAL_SHARE * _want[n])
    puzzles += [{**p, "pid": len(puzzles) + i, "n": n, "val": i >= _want[n] - n_val} for i, p in enumerate(_got[n])]
print(f"{len(puzzles)} puzzles (4-number {N_PUZZLES_4}, 3-number {N_PUZZLES_3}; validation "
      f"{sum(p['val'] for p in puzzles)}), disjoint from the test, SFT, GRPO and dump puzzles")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

dumps: 2560 completions, 160 groups, steps [np.int64(0), np.int64(20), np.int64(40), np.int64(60), np.int64(80), np.int64(100), np.int64(120), np.int64(140), np.int64(160), np.int64(180), np.int64(200), np.int64(220), np.int64(240), np.int64(260), np.int64(280), np.int64(300), np.int64(320), np.int64(340), np.int64(360), np.int64(380)]


README.md:   0%|          | 0.00/314 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.85MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/490364 [00:00<?, ? examples/s]

2000 puzzles (4-number 1000, 3-number 1000; validation 200), disjoint from the test, SFT, GRPO and dump puzzles


In [7]:
# [generate] Step 1: one normal completion per puzzle from the SFT policy (GRPO's prompt, temperature and cap). Step 2:
# for each usable normal completion (finished, its solution scored by the GRPO reward), each corruption of its
# reasoning, then the policy writes <solution> and <answer> after the forced "</reasoning>\n<solution>\n", with
# SOL_MAX_TOKENS of room, stopping at </answer>. swap takes the reasoning of another puzzle of the same size and split
# (a random order shifted by one: never its own). Saves NORMAL_PATH and CORRUPT_PATH (skipped when both exist). Then
# per HOT_TEMPERATURES: a new reasoning for the same puzzle at that temperature (to "</reasoning>", MAX_TOKENS), the
# solution from it as for the others (TEMPERATURE, SOL_MAX_TOKENS); usable only if the hot reasoning finished. Saves
# HOT_PATHS (skipped when it exists).
REASONING = re.compile(r"\s*<reasoning>\n?(.*?)\n?</reasoning>\s*(?=<solution>)", re.DOTALL)

def prefix(reasoning):
    """The forced start of the assistant turn, in the SFT traces' layout."""
    return "<reasoning>\n" + (f"{reasoning}\n" if reasoning else "") + "</reasoning>\n<solution>\n"

def cut(reasoning, frac):
    """The first frac of the reasoning's tokens, back to the last whole line ("" if no line fits)."""
    ids = tok(reasoning, add_special_tokens=False)["input_ids"]
    text = tok.decode(ids[:int(frac * len(ids))])
    return text[:text.rfind("\n")].rstrip() if "\n" in text else ""

def write_solutions(jobs):
    """The policy's <solution> and <answer> after each job's forced prefix; fills the job rows in place."""
    _ids = [chat_ids(countdown_prompt(json.loads(j["numbers"]), j["target"]))
            + tok(j["prefix"], add_special_tokens=False)["input_ids"] for j in jobs]
    _outs = llm.generate([{"prompt_token_ids": x} for x in _ids],
                         SamplingParams(n=1, temperature=TEMPERATURE, max_tokens=SOL_MAX_TOKENS, seed=SEED + 1,
                                        stop=["</answer>"], include_stop_str_in_output=True), lora_request=lora)
    assert all(list(o.prompt_token_ids) == x for o, x in zip(_outs, _ids)), "vLLM returned outputs for other prompts"
    for j, o in zip(jobs, _outs):
        full = j["prefix"] + o.outputs[0].text
        sol, ans = tag(full, "solution"), tag(full, "answer")
        j.update({"generated": o.outputs[0].text, "finished": o.outputs[0].text.rstrip().endswith("</answer>"),
                  "solution": sol, "answer": ans, "class": klass(sol, ans, json.loads(j["numbers"]), j["target"]),
                  "budget": SOL_MAX_TOKENS})
        j["usable"] = j["finished"] and scorable(sol, ans)

_have = os.path.exists(NORMAL_PATH) and os.path.exists(CORRUPT_PATH)
if not _have or not all(os.path.exists(p) for p in HOT_PATHS.values()):
    from vllm import LLM, SamplingParams
    from vllm.lora.request import LoRARequest
    llm = LLM(MODEL, dtype="bfloat16", gpu_memory_utilization=VLLM_GPU_MEMORY_UTILIZATION, max_model_len=4096,
              enable_prefix_caching=False, seed=SEED, enable_lora=True, max_lora_rank=16, max_loras=1)
    lora = LoRARequest("policy", 1, ADAPTER_PATH)
if _have:
    normal, corrupt = pd.read_csv(NORMAL_PATH), pd.read_csv(CORRUPT_PATH)
    SOL_MAX_TOKENS = int(corrupt.budget.iloc[0])
    print(f"[generate] skipped: {NORMAL_PATH} and {CORRUPT_PATH} exist (room for the solution: {SOL_MAX_TOKENS} tokens)")
else:
    _ids = [chat_ids(countdown_prompt(p["numbers"], p["target"])) for p in puzzles]
    _outs = llm.generate([{"prompt_token_ids": x} for x in _ids],
                         SamplingParams(n=1, temperature=TEMPERATURE, max_tokens=MAX_TOKENS, seed=SEED), lora_request=lora)
    assert all(list(o.prompt_token_ids) == x for o, x in zip(_outs, _ids)), "vLLM returned outputs for other prompts"
    rows = []
    for p, o in zip(puzzles, _outs):
        t = o.outputs[0].text
        m = REASONING.match(t)
        sol, ans = tag(t, "solution"), tag(t, "answer")
        i = t.find("<solution>\n", m.end()) if m else -1     # the span the corrupted ones generate: after "<solution>\n"
        j = t.find("</answer>", i) if i >= 0 else -1
        rows.append({"pid": p["pid"], "n": p["n"], "val": p["val"], "numbers": json.dumps(p["numbers"]),
                     "target": p["target"], "completion": t, "tokens": len(o.outputs[0].token_ids),
                     "truncated": o.outputs[0].finish_reason == "length", "reasoning": m.group(1).strip() if m else None,
                     "span_tokens": (len(tok(t[i + len("<solution>\n"):j + len("</answer>")], add_special_tokens=False)["input_ids"])
                                     if m and i >= 0 and j >= 0 else None),
                     "solution": sol, "answer": ans, "class": klass(sol, ans, p["numbers"], p["target"])})
    normal = pd.DataFrame(rows)
    normal["usable"] = [not tr and isinstance(r, str) and pd.notna(st) and scorable(s, a)
                        for tr, r, st, s, a in zip(normal.truncated, normal.reasoning, normal.span_tokens,
                                                   normal.solution, normal.answer)]
    if SOL_MAX_TOKENS is None:
        SOL_MAX_TOKENS = int(math.ceil(normal.loc[normal.usable, "span_tokens"].quantile(0.99) / 8) * 8)
    normal["usable"] &= normal.span_tokens.fillna(10**9) <= SOL_MAX_TOKENS   # the same room as the corrupted ones
    print(f"normal: {len(normal)} completions, usable {normal.usable.sum()} (truncated {normal.truncated.mean():.2f}); "
          f"room for solution + answer: {SOL_MAX_TOKENS} tokens (normal spans: median "
          f"{normal.span_tokens.median():.0f}, max {normal.span_tokens.max():.0f})")

    rng = random.Random(SEED + 1)
    use = normal[normal.usable]
    donor = {}
    for (_n, _v), d in use.groupby(["n", "val"]):
        order = list(d.index)
        rng.shuffle(order)
        donor.update({a: b for a, b in zip(order, order[1:] + order[:1])})
    jobs = []
    for i, r in use.iterrows():
        for kind in KINDS:
            reasoning = ("" if kind == "empty" else use.at[donor[i], "reasoning"] if kind == "swap"
                         else cut(r.reasoning, CUT_FRACTION))
            jobs.append({"pid": r.pid, "kind": kind, "donor_pid": int(use.at[donor[i], "pid"]) if kind == "swap" else None,
                         "reasoning_tokens": len(tok(reasoning, add_special_tokens=False)["input_ids"]),
                         "of_tokens": len(tok(r.reasoning, add_special_tokens=False)["input_ids"]),
                         "prefix": prefix(reasoning), "numbers": r.numbers, "target": r.target, "n": r.n, "val": r.val})
    write_solutions(jobs)
    corrupt = pd.DataFrame(jobs).drop(columns=["prefix"])
    os.makedirs(OUT_DIR, exist_ok=True)
    normal.to_csv(NORMAL_PATH, index=False)
    corrupt.to_csv(CORRUPT_PATH, index=False)
    print(f"saved {NORMAL_PATH} and {CORRUPT_PATH} ({len(corrupt)} corrupted): next, Consistency_Corrupt_Judge")

_use = normal[normal.usable]
for t, path in HOT_PATHS.items():
    if os.path.exists(path):
        print(f"[generate] skipped: {path} exists")
        continue
    _ids = [chat_ids(countdown_prompt(json.loads(r.numbers), r.target)) + tok("<reasoning>\n", add_special_tokens=False)["input_ids"]
            for r in _use.itertuples()]
    _outs = llm.generate([{"prompt_token_ids": x} for x in _ids],
                         SamplingParams(n=1, temperature=t, max_tokens=MAX_TOKENS, seed=SEED + 2,
                                        stop=["</reasoning>", "<solution>"]), lora_request=lora)
    assert all(list(o.prompt_token_ids) == x for o, x in zip(_outs, _ids)), "vLLM returned outputs for other prompts"
    jobs = []
    for r, o in zip(_use.itertuples(), _outs):
        reasoning = o.outputs[0].text.strip()
        jobs.append({"pid": r.pid, "kind": f"hot{t:g}", "donor_pid": None,
                     "reasoning_tokens": len(o.outputs[0].token_ids),
                     "of_tokens": len(tok(r.reasoning, add_special_tokens=False)["input_ids"]),
                     "hot_finished": o.outputs[0].finish_reason == "stop", "hot_reasoning": reasoning,
                     "prefix": prefix(reasoning), "numbers": r.numbers, "target": r.target, "n": r.n, "val": r.val})
    write_solutions(jobs)
    hot = pd.DataFrame(jobs).drop(columns=["prefix"])
    hot["usable"] &= hot.hot_finished
    hot.to_csv(path, index=False)
    print(f"saved {path}: hot reasoning finished {hot.hot_finished.mean():.3f}, usable {hot.usable.mean():.3f}")
corrupt = pd.concat([pd.read_csv(CORRUPT_PATH), *(pd.read_csv(p) for p in HOT_PATHS.values())], ignore_index=True)

(MainProcess pid=8579) INFO 10-26 23:15:02 [api_utils.py:286] non-default args: {'dtype': 'bfloat16', 'max_model_len': 4096, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.5, 'disable_log_stats': True, 'enable_lora': True, 'model': 'meta-llama/Llama-3.2-3B-Instruct'}
(MainProcess pid=8579) WARNING 10-26 23:15:02 [arg_utils.py:1910] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
(MainProcess pid=8579) INFO 10-26 23:15:19 [model.py:699] Resolved architecture: LlamaForCausalLM
(MainProcess pid=8579) INFO 10-26 23:15:19 [model.py:2109] Using max model len 4096
(MainProcess pid=8579) INFO 10-26 23:15:19 [scheduler.py:308] Chunked prefill is enabled with max_num_batched_tokens=8192.


model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Parse safetensors files:   0%|          | 0/2 [00:00<?, ?it/s]

(MainProcess pid=8579) INFO 10-26 23:15:21 [kernel.py:408] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

(MainProcess pid=8579) INFO 10-26 23:15:26 [core.py:129] Initializing a V1 LLM engine (v0.31.0) with config: model='meta-llama/Llama-3.2-3B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.2-3B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, aux_output_config=AuxOutputConfig(enable_return_routed_experts=False, max_bytes=None), kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=Ob

... (21 outputs left out)

Processed prompts: 100%|██████████| 1328/1328 [02:15<00:00,  9.82it/s, est. speed input: 1473.83 toks/s, output: 8401.35 toks/s]


Rendering prompts:   0%|          | 0/1328 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1328/1328 [01:00<00:00, 21.96it/s, est. speed input: 22330.18 toks/s, output: 1024.86 toks/s] 


saved /content/drive/MyDrive/Consistency/countdown/sft/Llama-3.2-3B-Instruct-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-judge500-mix3x500/judge_corrupt_empty-swap-cut_cut0.5_n4x1000_n3x1000_t0.9_seed0/corrupt_hot1.5.csv: hot reasoning finished 0.204, usable 0.203


In [8]:
# [label gap] Step 2 check (the checker measures, nothing trains on it): per size, what the normal and the corrupted
# reasoning give: usable share (finished, solution scored by the GRPO reward), accuracy and kinds of answer among the
# usable ones. Then per pair (normal vs corrupted, same puzzle, both usable): the label is right when the normal is
# correct and the corrupted wrong, inverted in the opposite case, and says nothing about correctness when both agree
# (both wrong: is the corrupted one more often a fake?). Stop here if right is not well above inverted.

for _d in (normal, corrupt):
    _m = _d.usable & _d.answer.isna()
    _d.loc[_m, "answer"] = "None"   # pandas read the answer "None" as missing

CLASSES = ["correct", "fake", "wrong numbers", "honest miss", "bad answer", "other"]
both = pd.concat([normal.assign(kind="normal"), corrupt], ignore_index=True)
rows = []
for (n, kind), d in both.groupby(["n", "kind"], sort=False):
    u = d[d.usable]
    rows.append({"n": n, "source": kind, "rows": len(d), "usable": d.usable.mean(), "accuracy": (u["class"] == "correct").mean(),
                 **{c: (u["class"] == c).mean() for c in CLASSES[1:]},
                 "reasoning kept": (d.reasoning_tokens / d.of_tokens).mean() if kind == "cut" else np.nan})
gap = pd.DataFrame(rows).set_index(["n", "source"]).sort_index()
with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 200):
    print("by source (accuracy and kinds among the usable rows):\n" + gap.to_string())

_nu = normal[normal.usable].set_index("pid")
pairs_all = corrupt[corrupt.usable].join(_nu[["solution", "answer", "class"]], on="pid", rsuffix="_normal")
pairs_all["identical"] = [judge_text(a, b) == judge_text(c, d) for a, b, c, d in
                          zip(pairs_all.solution, pairs_all.answer, pairs_all.solution_normal, pairs_all.answer_normal)]
rows = []
for (n, kind), d in pairs_all.groupby(["n", "kind"], sort=False):
    nc, cc = d.class_normal == "correct", d["class"] == "correct"
    bw = d[~nc & ~cc]
    rows.append({"n": n, "kind": kind, "pairs": len(d), "identical": d.identical.mean(), "right": (nc & ~cc).mean(),
                 "inverted": (~nc & cc).mean(), "both correct": (nc & cc).mean(), "both wrong": len(bw) / len(d),
                 "both wrong: fake normal": (bw.class_normal == "fake").mean(),
                 "both wrong: fake corrupted": (bw["class"] == "fake").mean()})
lab = pd.DataFrame(rows).set_index(["n", "kind"]).sort_index()
with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 200):
    print("\npairs (normal vs corrupted, same puzzle, both usable):\n" + lab.to_string())
gap.to_csv(f"{OUT_DIR}/label_gap_sources.csv")
lab.to_csv(f"{OUT_DIR}/label_gap_pairs.csv")

by source (accuracy and kinds among the usable rows):
          rows  usable  accuracy  fake  wrong numbers  honest miss  bad answer  other  reasoning kept
n source                                                                                             
3 cut      821   0.976     0.029 0.683          0.196        0.079       0.002  0.011           0.465
  empty    821   0.976     0.210 0.366          0.408        0.007       0.001  0.007             NaN
  hot1.3   821   0.714     0.613 0.244          0.108        0.009       0.002  0.026             NaN
  hot1.5   821   0.295     0.645 0.285          0.050        0.008       0.008  0.004             NaN
  normal  1000   0.821     0.597 0.177          0.213        0.006       0.000  0.007             NaN
  swap     821   0.998     0.000 0.189          0.801        0.000       0.000  0.010             NaN
4 cut      507   0.970     0.083 0.593          0.124        0.167       0.014  0.018           0.469
  empty    507   0.846     0

In [9]:
# [release] All results are on Drive: release the runtime if UNASSIGN_WHEN_DONE (a failed cell stops Run all before
# this one, and the runtime stays up for a look).
if UNASSIGN_WHEN_DONE:
    from google.colab import runtime
    runtime.unassign()